# 00 - Container Setup

This notebook launches your **personal** ALCHEMI BGR NIM, Prometheus, and Grafana containers on random ports so multiple participants can share one lab VM without colliding on the standard 8000/9090/3000 ports.

**Run this notebook once at the start of the workshop**, then open [`alchemi-oer-catalyst-screening.ipynb`](alchemi-oer-catalyst-screening.ipynb) to begin the tutorial.

### Prerequisites (lab admin has already done this)

- `docker load -i alchemi-bundle.tar` on the lab VM, which loads four images:
  - `alchemi-playbook:latest` (this JupyterLab environment)
  - `nvcr.io/nim/nvidia/alchemi-bgr:1.0.0`
  - `prom/prometheus:latest`
  - `grafana/grafana:latest`
- JupyterLab launched via `build/run-jupyter.sh`, which mounts the host Docker socket so this notebook can spawn sibling containers.

---
## 1. NGC API Key

The BGR NIM uses your NGC API key at runtime to fetch model artifacts. Get one at [build.nvidia.com](https://build.nvidia.com) if you don't have it yet, then run the cell below and paste it into the prompt.

In [ ]:
import os
import getpass

if not os.environ.get("NGC_API_KEY"):
    os.environ["NGC_API_KEY"] = getpass.getpass("Enter your NGC API key: ")

print(f"NGC_API_KEY set (length = {len(os.environ['NGC_API_KEY'])})")

---
## 2. Verify Pre-loaded Images

All four images should already be present locally (loaded from `alchemi-bundle.tar` by the lab admin). If any are missing, this cell will tell you which one — flag it to the workshop staff before continuing.

In [ ]:
import subprocess

REQUIRED_IMAGES = [
    "alchemi-playbook:latest",
    "nvcr.io/nim/nvidia/alchemi-bgr:1.0.0",
    "prom/prometheus:latest",
    "grafana/grafana:latest",
]

out = subprocess.run(
    ["docker", "images", "--format", "{{.Repository}}:{{.Tag}}"],
    capture_output=True, text=True, check=True,
)
present = set(out.stdout.strip().splitlines())

missing = []
for img in REQUIRED_IMAGES:
    ok = img in present
    print(f"  {'OK ' if ok else 'NO '} {img}")
    if not ok:
        missing.append(img)

if missing:
    raise RuntimeError(
        f"Missing {len(missing)} image(s). Ask the lab admin to run "
        f"`docker load -i alchemi-bundle.tar`."
    )

---
## 3. NIM Cache

The BGR NIM downloads model profiles into a local cache on first start. We reuse a host-side directory so artifacts survive container restarts.

`HOST_NIM_CACHE` is the path on the lab VM host; `LOCAL_NIM_CACHE` is the same directory bind-mounted into this JupyterLab container. The setup notebook needs the host path to forward to the sibling BGR container.

In [ ]:
HOST_NIM_CACHE = os.environ.get("HOST_NIM_CACHE", os.path.expanduser("~/.cache/nim"))
LOCAL_NIM_CACHE = "/root/.cache/nim"
os.makedirs(LOCAL_NIM_CACHE, exist_ok=True)

print(f"Host:      {HOST_NIM_CACHE}")
print(f"Container: {LOCAL_NIM_CACHE}")

---
## 4. Allocate Per-Participant Ports

Each participant gets three random free ports in the 11000-11999 range. The choices are written to `config.py`, which the screening notebook imports.

In [ ]:
import random
import socket

def find_free_port(start=11000, end=11999, exclude=()):
    tried = set(exclude)
    for _ in range(200):
        port = random.randint(start, end)
        if port in tried:
            continue
        tried.add(port)
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            try:
                s.bind(("0.0.0.0", port))
                return port
            except OSError:
                continue
    raise RuntimeError(f"No free port in [{start}, {end}] after 200 attempts")

BGR_PORT = find_free_port()
PROMETHEUS_PORT = find_free_port(exclude=[BGR_PORT])
GRAFANA_PORT = find_free_port(exclude=[BGR_PORT, PROMETHEUS_PORT])

with open("config.py", "w") as f:
    f.write(
        "# Auto-generated by 00_container_setup.ipynb. Do not edit by hand.\n"
        f"BGR_PORT = {BGR_PORT}\n"
        f"PROMETHEUS_PORT = {PROMETHEUS_PORT}\n"
        f"GRAFANA_PORT = {GRAFANA_PORT}\n"
        f'BGR_SERVER = "http://host.docker.internal:{BGR_PORT}"\n'
    )

print(f"BGR_PORT        = {BGR_PORT}")
print(f"PROMETHEUS_PORT = {PROMETHEUS_PORT}")
print(f"GRAFANA_PORT    = {GRAFANA_PORT}")
print("Wrote config.py")

---
## 5. Render Prometheus and Grafana Configs

Both monitoring containers need their config files to point at the random ports allocated above. We render them into `.runtime/` and bind-mount the host-side path into the sibling containers.

In [ ]:
from pathlib import Path

RUNTIME_DIR = Path(".runtime")
(RUNTIME_DIR / "grafana-datasources").mkdir(parents=True, exist_ok=True)

prometheus_yml = (
    "global:\n"
    "  scrape_interval: 15s\n"
    "\n"
    "scrape_configs:\n"
    "  - job_name: 'nim-bgr'\n"
    "    metrics_path: '/v1/metrics'\n"
    "    static_configs:\n"
    f"      - targets: ['host.docker.internal:{BGR_PORT}']\n"
)
(RUNTIME_DIR / "prometheus.yml").write_text(prometheus_yml)

grafana_ds_yml = (
    "apiVersion: 1\n"
    "\n"
    "datasources:\n"
    "  - name: Prometheus\n"
    "    type: prometheus\n"
    "    access: proxy\n"
    f"    url: http://host.docker.internal:{PROMETHEUS_PORT}\n"
    "    isDefault: true\n"
)
(RUNTIME_DIR / "grafana-datasources" / "prometheus.yml").write_text(grafana_ds_yml)

HOST_REPO_ROOT = os.environ["HOST_REPO_ROOT"]
HOST_RUNTIME_DIR = f"{HOST_REPO_ROOT}/part-1-nim/.runtime"

print(f"Wrote {RUNTIME_DIR / 'prometheus.yml'}")
print(f"Wrote {RUNTIME_DIR / 'grafana-datasources/prometheus.yml'}")
print(f"Host-side runtime dir for bind-mounts: {HOST_RUNTIME_DIR}")

---
## 6. Launch the BGR NIM

Spawned as a sibling container on the lab VM host via the mounted docker socket. The port-suffixed name (`alchemi-bgr-<port>`) keeps multiple participants on one VM from colliding.

In [ ]:
BGR_NAME = f"alchemi-bgr-{BGR_PORT}"

subprocess.run(["docker", "rm", "-f", BGR_NAME], capture_output=True)

cmd = [
    "docker", "run", "-d",
    "--name", BGR_NAME,
    "--gpus", "all",
    "--shm-size", "8g",
    "--platform", "linux/amd64",
    "--add-host=host.docker.internal:host-gateway",
    "-p", f"{BGR_PORT}:8000",
    "-e", f"NGC_API_KEY={os.environ['NGC_API_KEY']}",
    "-e", "ALCHEMI_NIM_MODEL_TYPE=mace",
    "-e", "ALCHEMI_NIM_PBC=true",
    "-e", "ALCHEMI_NIM_BGR_OPTIMIZER_PRESET=materials",
    "-e", "ALCHEMI_NIM_DFT3_ENABLED=true",
    "-v", f"{HOST_NIM_CACHE}:/home/nvs/.cache/nim",
    "nvcr.io/nim/nvidia/alchemi-bgr:1.0.0",
]
out = subprocess.run(cmd, capture_output=True, text=True, check=True)
print(f"Started {BGR_NAME} -> port {BGR_PORT}  (id: {out.stdout.strip()[:12]})")

---
## 7. Launch Prometheus and Grafana

In [ ]:
PROM_NAME = f"alchemi-prom-{PROMETHEUS_PORT}"
GRAFANA_NAME = f"alchemi-grafana-{GRAFANA_PORT}"

subprocess.run(["docker", "rm", "-f", PROM_NAME], capture_output=True)
subprocess.run(["docker", "rm", "-f", GRAFANA_NAME], capture_output=True)

prom_cmd = [
    "docker", "run", "-d",
    "--name", PROM_NAME,
    "--add-host=host.docker.internal:host-gateway",
    "-p", f"{PROMETHEUS_PORT}:9090",
    "-v", f"{HOST_RUNTIME_DIR}/prometheus.yml:/etc/prometheus/prometheus.yml:ro",
    "prom/prometheus:latest",
]
subprocess.run(prom_cmd, capture_output=True, text=True, check=True)
print(f"Started {PROM_NAME} -> port {PROMETHEUS_PORT}")

grafana_cmd = [
    "docker", "run", "-d",
    "--name", GRAFANA_NAME,
    "--add-host=host.docker.internal:host-gateway",
    "-p", f"{GRAFANA_PORT}:3000",
    "-e", "GF_SECURITY_ADMIN_PASSWORD=admin",
    "-v", f"{HOST_RUNTIME_DIR}/grafana-datasources:/etc/grafana/provisioning/datasources:ro",
    "grafana/grafana:latest",
]
subprocess.run(grafana_cmd, capture_output=True, text=True, check=True)
print(f"Started {GRAFANA_NAME} -> port {GRAFANA_PORT}")

---
## 8. Wait for BGR Ready

BGR takes 1-3 minutes to load the MACE-MP-0 model into GPU memory. We poll the readiness endpoint until it returns 200.

In [ ]:
import time
import urllib.request
import urllib.error

URL = f"http://host.docker.internal:{BGR_PORT}/v1/health/ready"
TIMEOUT_S = 600
start = time.time()

print(f"Polling {URL} ...")
while True:
    elapsed = int(time.time() - start)
    if elapsed > TIMEOUT_S:
        raise RuntimeError(
            f"BGR not ready after {TIMEOUT_S}s. "
            f"Check logs with: !docker logs {BGR_NAME}"
        )
    try:
        with urllib.request.urlopen(URL, timeout=2) as resp:
            if resp.status == 200:
                print(f"BGR ready after {elapsed}s")
                break
    except (urllib.error.URLError, ConnectionError, TimeoutError):
        pass
    time.sleep(5)

---
## 9. Connection Details

Two URL forms for each service:
- **Notebook URL** uses `host.docker.internal` because notebook code runs inside this JupyterLab container.
- **Browser URL** uses `localhost` because your browser runs on the lab VM host (or your laptop with port-forwarding to the host).

In [ ]:
print("\nEndpoints")
print("---------")
print(f"  BGR (notebook):  http://host.docker.internal:{BGR_PORT}")
print(f"  BGR (browser):   http://localhost:{BGR_PORT}/v1/health/ready")
print(f"  Prometheus:      http://localhost:{PROMETHEUS_PORT}")
print(f"  Grafana:         http://localhost:{GRAFANA_PORT}    (admin / admin)")
print("")
print("Next: open alchemi-oer-catalyst-screening.ipynb.")

---
## Teardown (run at the end of the workshop)

Stop and remove your three sibling containers. Uncomment the line below before running.

In [ ]:
# subprocess.run(["docker", "rm", "-f", BGR_NAME, PROM_NAME, GRAFANA_NAME], check=False)